# nb_02_create_search_index — define the Azure AI Search index

Idempotently creates (or updates) the vector index used for RAG. Fields include a vector
column, page numbers, and `allowed_groups` for **query-time security trimming**.

Run once, and again whenever the index schema changes. See `PRODUCT_SPEC.md` sections 7.2, 10, 13.


## Config + secret
The Search admin key is read from Key Vault via `mssparkutils` — never hard-coded.


In [ ]:
cfg = {r['key']: r['value'] for r in spark.table('config').collect()}
SEARCH_ENDPOINT = cfg['search_endpoint']
INDEX_NAME      = cfg['search_index_name']
DIMENSIONS      = int(cfg['embedding_dimensions'])

# Key Vault-backed secret (configure kv_name / kv_search_admin_key_secret in the config table).
SEARCH_ADMIN_KEY = mssparkutils.credentials.getSecret(
    f"https://{cfg['kv_name']}.vault.azure.net/", cfg['kv_search_admin_key_secret'])
print('endpoint:', SEARCH_ENDPOINT, '| index:', INDEX_NAME, '| dims:', DIMENSIONS)


## Install SDK (if not already available on the pool)


In [ ]:
# %pip install azure-search-documents==11.5.1


## Define + create/update the index


In [ ]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex, SearchField, SearchFieldDataType, SimpleField, SearchableField,
    VectorSearch, HnswAlgorithmConfiguration, VectorSearchProfile,
    SemanticConfiguration, SemanticPrioritizedFields, SemanticField, SemanticSearch,
)

client = SearchIndexClient(SEARCH_ENDPOINT, AzureKeyCredential(SEARCH_ADMIN_KEY))

fields = [
    SimpleField(name='chunk_id', type=SearchFieldDataType.String, key=True),
    SimpleField(name='file_path', type=SearchFieldDataType.String, filterable=True),
    SimpleField(name='file_name', type=SearchFieldDataType.String, filterable=True, sortable=True),
    SimpleField(name='file_extension', type=SearchFieldDataType.String, filterable=True, facetable=True),
    SearchableField(name='content', type=SearchFieldDataType.String),
    SearchField(
        name='content_vector',
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True, vector_search_dimensions=DIMENSIONS,
        vector_search_profile_name='vprofile'),
    SimpleField(name='page_number', type=SearchFieldDataType.Int32, filterable=True, sortable=True),
    SimpleField(name='chunk_index', type=SearchFieldDataType.Int32, sortable=True),
    # Security trimming: the Entra group GUIDs allowed to see this chunk.
    SimpleField(name='allowed_groups', type=SearchFieldDataType.Collection(SearchFieldDataType.String),
                filterable=True),
    SimpleField(name='embedding_model', type=SearchFieldDataType.String, filterable=True),
    SimpleField(name='chunk_strategy_version', type=SearchFieldDataType.String, filterable=True),
    SimpleField(name='indexed_utc', type=SearchFieldDataType.DateTimeOffset, filterable=True, sortable=True),
]

vector_search = VectorSearch(
    algorithms=[HnswAlgorithmConfiguration(name='hnsw')],
    profiles=[VectorSearchProfile(name='vprofile', algorithm_configuration_name='hnsw')],
)

semantic = SemanticSearch(configurations=[
    SemanticConfiguration(
        name='default',
        prioritized_fields=SemanticPrioritizedFields(
            content_fields=[SemanticField(field_name='content')])),
])

index = SearchIndex(name=INDEX_NAME, fields=fields,
                    vector_search=vector_search, semantic_search=semantic)

result = client.create_or_update_index(index)
print('index ready:', result.name)


## Query-time security trimming (reference)
The consuming app filters on the caller's Entra group membership, e.g.:

```
filter = "allowed_groups/any(g: search.in(g, '<comma-separated-user-group-guids>'))"
```

How the app obtains the caller's groups (e.g. OBO) is an open question — see spec §13.
